<a href="https://colab.research.google.com/github/ahlqui/VeloxChemColabs/blob/main/AtomicCharges.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>




# Calculate partial charges


This notebook requires that you have a molecular structure. Two options:

Define Molecule as XYZ-coordinates: Can be generatred with free software such as www.avogadro.cc

Define Molecule with SMILES code, A molecule can be defined using a SMILES code (example below). We have two suggested ways to generate smiles from structure. 1) Sketch your molecule at https://www.rcsb.org/chemical-sketch and the SMILES code will be shown right below the structure. 2) Build your molecule at https://molview.org/ , go to Tools/Information Card and it will show you the SMILES code. Just copy/paste it into the SMILES box below.

Example of xyz-coordinates for CO2:

O 0.00 0.00 0.00

C 0.00 0.00 1.20

O 0.00 0.00 2.40

Example of SMILES for CO2:

O=C=O

In [17]:
# Local environment check for QuimicaAutomocio P2.
# The original Colab version used a Colab-only installation cell.
# In local Jupyter, skip that installation cell and use the conda environment instead:
#   conda activate quimica-echem

import sys
try:
    import veloxchem as vlx
    print(f"VeloxChem imported correctly: {getattr(vlx, '__version__', 'unknown')}")
    print(f"Python executable: {sys.executable}")
except Exception as exc:
    raise RuntimeError(
        "VeloxChem is not available in this kernel. Activate the conda environment "
        "with `conda activate quimica-echem`, start Jupyter from that terminal, "
        "and select the quimica-echem kernel."
    ) from exc

VeloxChem imported correctly: 1.0rc4
Python executable: c:\Users\Nayanika\miniconda3\envs\quimica-echem\python.exe


In [18]:
xyz_coordinates = """
C             -2.009286891018         1.541982466480        -0.091237472346
C             -2.183437552991         0.488041320387         1.006309055321
C             -2.080814838236        -0.972504794947         0.534418746164
C             -0.663251349548        -1.588311841902         0.424421199251
C              0.047533431840        -1.729714070872         1.782207035355
C             -0.652713363584        -2.675792283892         2.757950461594
O             -0.924588179020        -3.950577870131         2.209678918635
C              0.183938290209        -0.938422542378        -0.640768217645
C              1.156225356327         0.040946965219        -0.542892373696
C              1.723346475839         0.778721012517         0.631182961422
N              1.613640131739         0.351199733726        -1.816215263948
C              0.958655455701        -0.409736899680        -2.754052118952
C              1.080609659362        -0.453026853424        -4.147776492868
C              0.262757889140        -1.343792457860        -4.838305680541
C             -0.652742932751        -2.173033177675        -4.155547582872
C             -0.768459751558        -2.126994943742        -2.769769644692
C              0.044213913963        -1.236166661102        -2.045038221694
H             -2.753081178313         1.395897269705        -0.891976036837
H             -2.149088179107         2.559286186835         0.309984578211
H             -1.017186380192         1.494827647548        -0.560816583391
H             -3.176514821392         0.631566546753         1.466532904159
H             -1.462458525063         0.673201021412         1.822846225720
H             -2.669252903436        -1.612959630936         1.211848247471
H             -2.571215122030        -1.062117643307        -0.449853439052
H             -0.833557121019        -2.624027893949         0.085088616989
H              1.072706141814        -2.101378642675         1.599857896302
H              0.167092862955        -0.755959937679         2.283580778456
H             -1.629621423235        -2.265054082309         3.058390888446
H             -0.047807295007        -2.753825647038         3.684531460790
H             -0.086352219196        -4.342883444946         1.930205500954
H              2.371470820936         0.140892168559         1.253640652911
H              2.328353864498         1.635827497990         0.296675908402
H              0.930285398899         1.175342682311         1.281496606668
H              2.335060600426         1.030594497289        -2.014125676658
H              1.790429055523         0.188208579535        -4.676404115560
H              0.331264175866        -1.401721715899        -5.927369643136
H             -1.278843094364        -2.861504585824        -4.728350649020
H             -1.479479469859        -2.775226236501        -2.251854595749
"""


In [19]:
import veloxchem as vlx

molecule = vlx.Molecule.read_str(xyz_coordinates)
print('Structure of the molecule entered: ')
molecule.show(atom_indices=True)

Structure of the molecule entered: 


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [20]:
#@title DFT single point and charge calculation
#@markdown For classroom tests, start with a small basis such as STO-3G or DEF2-SVP.


#molecule.set_charge(1)

basis_set = 'DEF2-SVP' #@param  ['STO-3G', 'DEF2-SV(P)', 'DEF2-SVP', 'DEF2-SVPD', 'DEF2-TZVP']
basis = vlx.MolecularBasis.read(molecule, basis_set)
scf_drv = vlx.ScfRestrictedDriver()
method_dict = {'conv_thresh': 1e-4}
scf_drv.update_settings(method_dict)
mute_output = True # @param {type:"boolean"}
if mute_output:
    scf_drv.ostream.mute()
functional = 'B3LYP' #@param ['SLATER', 'BLYP', 'B3LYP', 'PBE', 'PBE0', 'BP86']
scf_drv.xcfun = functional
scf_results = scf_drv.compute(molecule, basis)

# VeloxChem 1.0 separates ESP and RESP drivers. Older Colab notebooks used
# RespChargesDriver(..., "esp"), which is now deprecated and raises an error.
esp_drv = vlx.EspChargesDriver()
esp_drv.ostream.mute()
esp_charges = esp_drv.compute(molecule, basis, scf_results)

resp_drv = vlx.RespChargesDriver()
resp_drv.ostream.mute()
resp_charges = resp_drv.compute(molecule, basis, scf_results)

chelpg_drv = vlx.EspChargesDriver()
chelpg_drv.ostream.mute()
chelpg_drv.grid_type = "chelpg"
chelpg_drv.equal_charges = "1=1, 1=1"
chelpg_charges = chelpg_drv.compute(molecule, basis, scf_results)

for title, charges in [
    ("ESP charge", esp_charges),
    ("RESP charge", resp_charges),
    ("CHELPG charge", chelpg_charges),
]:
    print(f"Atom      {title}")
    print(20 * "-")
    for label, charge in zip(molecule.get_labels(), charges):
        print(f"{label :s} {charge : 18.6f}")
    print(20 * "-")
    print(f"Total: {charges.sum() : 13.6f}\n")

Atom      ESP charge
--------------------
C          -0.243208
C           0.299021
C          -0.427089
C           0.575192
C          -0.415045
C           0.336256
O          -0.624257
C          -0.629960
C           0.381463
C          -0.380752
N          -0.432923
C           0.130050
C          -0.250316
C          -0.111854
C          -0.168111
C          -0.223564
C           0.248071
H           0.061043
H           0.032164
H           0.067235
H          -0.036792
H          -0.075921
H           0.102504
H           0.090709
H          -0.044673
H           0.093352
H           0.091605
H           0.013541
H          -0.031807
H           0.390767
H           0.106887
H           0.093556
H           0.132929
H           0.337686
H           0.141865
H           0.113195
H           0.121186
H           0.135994
--------------------
Total:     -0.000000

Atom      RESP charge
--------------------
C          -0.034333
C           0.000116
C          -0.021181
C          

In [21]:
#@title Visualize the charges on molecule
import py3Dmol
viewer = py3Dmol.view(linked=False, width=500, height=500)

charge_type = 'ESP' # @param ["RESP", "ESP", "CHELPG"]
charge_map = {
    'ESP': esp_charges,
    'RESP': resp_charges,
    'CHELPG': chelpg_charges,
}
charges = charge_map[charge_type]
print(f'{charge_type} charges:')
viewer.addModel(molecule.get_xyz_string(), "xyz")
for i, charge in enumerate(charges):
    viewer.addLabel(
        "{:.2f}".format(charge),
        {'backgroundOpacity': 0, 'fontColor': 'black', 'backgroundColor': 'white'},
        {'index': i},
    )
viewer.setStyle({"stick": {}, "sphere": {"radius": 0.4}})
viewer.rotate(-45, "y")
viewer.zoomTo()
viewer.show()

ESP charges:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.